# NB07 — Machine Learning: Volatility Forecasting

**Target**: Forward realized volatility at 5-day and 21-day horizons (Yang-Zhang estimator)

**Feature Set**: Lagged realized vol, GARCH conditional vol, returns, momentum, volume,
macro factors, regime state, sentiment scores (t-1 lagged from NB06)

**Models** (all with walk-forward expanding-window evaluation):
1. Ridge Regression — linear baseline with L2 regularization
2. Lasso Regression — linear baseline with L1 regularization (feature selection)
3. Random Forest — 500 trees, non-linear ensemble
4. XGBoost — gradient boosting with Optuna hyperparameter tuning
5. LightGBM — alternative gradient boosting for comparison
6. KNN Regression — local non-parametric benchmark
7. Stacking Ensemble — Ridge meta-learner on base model predictions

**Walk-Forward Protocol**: Expanding window, retrain every 63 trading days (quarterly).
Initial training: first 70% of data. Consistent with NB08 for valid comparison in NB10.

**CRITICAL**: Scaler INSIDE pipeline. No SMOTE (regression, not classification).

**Dependencies**: NB01 (master data), NB03 (conditional vol), NB05 (regime labels), NB06 (sentiment)

**Output**: `vol_forecast_predictions.parquet`, `model_comparison_table.csv`, SHAP values

In [1]:
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.config import *
from src.feature_engineering import (
    compute_log_returns, realized_vol, yang_zhang_vol,
    rsi, bollinger_pct_b, rate_of_change, volume_zscore, rolling_beta
)
from src.ml_pipeline import (
    make_regression_pipeline, walk_forward_predict, regression_metrics,
    diebold_mariano_test, mincer_zarnowitz_test, build_stacking_ensemble,
    optuna_xgboost_regression
)
from src.visualization import save_fig, plot_model_comparison
import logging
logging.basicConfig(level=logging.INFO)
print('Imports OK')

Imports OK


## 1. Feature Assembly

In [2]:
# Load all upstream data
master = pd.read_parquet(MASTER_DATA_FILE)
adj_tickers = [t for t in TICKERS if t in master.columns]
log_ret = compute_log_returns(master[adj_tickers])
print(f'Master: {master.shape}, Tickers: {len(adj_tickers)}')

# Conditional volatility from NB03 (optional — enhances features if available)
cond_vol = pd.read_parquet(COND_VOL_FILE) if COND_VOL_FILE.exists() else pd.DataFrame()
print(f'GARCH cond vol: {"loaded " + str(cond_vol.shape) if not cond_vol.empty else "not available (run NB03 first)"}')

# Regime labels from NB05 (optional)
regime = pd.read_parquet(REGIME_LABELS_FILE) if REGIME_LABELS_FILE.exists() else pd.DataFrame()
print(f'Regime labels: {"loaded " + str(regime.shape) if not regime.empty else "not available (run NB05 first)"}')

# Sentiment features from NB06 (optional, t-1 lagged)
sentiment = pd.read_parquet(SENTIMENT_FILE) if SENTIMENT_FILE.exists() else pd.DataFrame()
print(f'Sentiment: {"loaded " + str(sentiment.shape) if not sentiment.empty else "not available (run NB06 first)"}')

# SPY returns for beta computation
spy_col = [c for c in master.columns if 'SPY' in c]
spy_ret = compute_log_returns(master[[spy_col[0]]])[spy_col[0]] if spy_col else None

Master: (2526, 33), Tickers: 20
GARCH cond vol: loaded (2521, 20)
Regime labels: not available (run NB05 first)
Sentiment: loaded (45368, 5)


## 2. Feature Engineering (Per-Ticker)

Build a rich feature set for each ticker combining:
- Lagged realized vol (5d, 21d, 63d)
- GARCH conditional vol (from NB03)
- Returns and momentum indicators (RSI, Bollinger %B, ROC)
- Volume features (z-score)
- Cross-asset beta (vs SPY)
- Regime state and probability (from NB05)
- Sentiment features (t-1 lagged from NB06)

In [3]:
def build_vol_features(ticker, horizon=5):
    """Build feature matrix X and target y for volatility forecasting."""
    prices = master[ticker].dropna()
    log_r = np.log(prices / prices.shift(1))

    # Target: forward realized vol (annualized)
    fwd_vol = log_r.rolling(horizon).std().shift(-horizon) * np.sqrt(252)

    X = pd.DataFrame(index=prices.index)

    # Lagged realized vol at multiple horizons
    for w in [5, 21, 63]:
        X[f'realized_vol_{w}d'] = log_r.rolling(w).std() * np.sqrt(252)

    # Returns features
    X['log_return'] = log_r
    X['abs_return'] = log_r.abs()
    X['return_5d'] = prices.pct_change(5)
    X['return_21d'] = prices.pct_change(21)

    # Momentum
    X['rsi_14'] = rsi(prices, 14)
    X['bollinger_pctb'] = bollinger_pct_b(prices)
    X['roc_20d'] = rate_of_change(prices, 20)

    # Volume
    if ticker in master.columns:
        # Volume z-score requires raw volume — use realized vol as proxy
        X['vol_ratio_5_21'] = X['realized_vol_5d'] / X['realized_vol_21d'].clip(lower=1e-6)

    # GARCH conditional vol (from NB03)
    if not cond_vol.empty and ticker in cond_vol.columns:
        X['garch_cond_vol'] = cond_vol[ticker].reindex(prices.index)

    # Cross-asset beta
    if spy_ret is not None:
        X['beta_spy_63d'] = rolling_beta(log_r, spy_ret, window=63)

    # Regime features (from NB05)
    if not regime.empty and 'regime_state' in regime.columns:
        X['regime_state'] = regime['regime_state'].reindex(prices.index)
        for col in regime.columns:
            if col.startswith('regime_prob_'):
                X[col] = regime[col].reindex(prices.index)

    # Sentiment features (from NB06, already t-1 lagged)
    if not sentiment.empty:
        ticker_sent = sentiment[sentiment['ticker'] == ticker].drop(columns=['ticker'], errors='ignore')
        for col in ticker_sent.columns:
            X[f'sent_{col}'] = ticker_sent[col].reindex(prices.index)

    # Clean up
    X = X.dropna()
    y = fwd_vol.reindex(X.index)
    valid = ~(X.isna().any(axis=1) | y.isna())
    return X[valid], y[valid]

# Build features for demonstration ticker
X_demo, y_demo = build_vol_features('NVDA', horizon=5)
print(f'NVDA features: {X_demo.shape[1]} features, {X_demo.shape[0]} observations')
print(f'Feature list: {list(X_demo.columns)}')

NVDA features: 17 features, 2330 observations
Feature list: ['realized_vol_5d', 'realized_vol_21d', 'realized_vol_63d', 'log_return', 'abs_return', 'return_5d', 'return_21d', 'rsi_14', 'bollinger_pctb', 'roc_20d', 'vol_ratio_5_21', 'garch_cond_vol', 'beta_spy_63d', 'sent_sentiment_mean', 'sent_sentiment_std', 'sent_sentiment_volume', 'sent_sentiment_momentum']


## 2b. Embargo/Purge Protocol & Feature Selection

**Embargo/Purge** (Lopez de Prado, 2018, Ch.7): When predicting h-day forward
targets, observations near the train/test boundary have labels that overlap with
test-period returns. We purge the last h training observations and embargo h days
after the train end.

**Feature Selection** (pre-modeling):
1. VIF filter: remove features with VIF > 10 (multicollinearity)
2. Stability Selection: Lasso on 100 random 50% subsamples, retain features selected >60% of runs
3. Compare full vs. filtered feature sets on OOS RMSE

In [4]:
from src.ml_pipeline import vif_filter, stability_selection, feature_selection_comparison

# ── VIF Filter ──
X_demo_vif = vif_filter(X_demo, threshold=10.0)
print(f'VIF filter: {X_demo.shape[1]} → {X_demo_vif.shape[1]} features')
dropped_vif = set(X_demo.columns) - set(X_demo_vif.columns)
if dropped_vif:
    print(f'  Dropped (VIF>10): {dropped_vif}')

# ── Stability Selection ──
stable_features = stability_selection(
    X_demo, y_demo, n_bootstrap=100, threshold=0.6, random_state=RANDOM_STATE
)
print(f'\nStability Selection: {len(stable_features)}/{X_demo.shape[1]} features retained (>60% selection)')
print(f'  Stable features: {stable_features}')

# ── Feature Selection Comparison ──
# Split for comparison
split = int(len(X_demo) * TRAIN_RATIO)
X_tr, X_te = X_demo.iloc[:split], X_demo.iloc[split:]
y_tr, y_te = y_demo.iloc[:split], y_demo.iloc[split:]

comparison_fs = feature_selection_comparison(
    X_tr, y_tr, X_te, y_te,
    pipeline_factory=lambda: make_regression_pipeline('xgboost')
)
print('\n--- Feature Selection Comparison (XGBoost, NVDA) ---')
print(comparison_fs.to_string(index=False))

print(f'\n--- Embargo/Purge Note ---')
print(f'For h=5d forecast: purge last 5 training obs + embargo 5 days after train end')
print(f'For h=21d forecast: purge last 21 training obs + embargo 21 days')
print(f'Applied inside walk_forward_predict() via embargo_days parameter')

INFO:src.ml_pipeline:VIF filter: dropped garch_cond_vol (VIF=123.8)
INFO:src.ml_pipeline:VIF filter: dropped rsi_14 (VIF=76.5)
INFO:src.ml_pipeline:VIF filter: dropped realized_vol_21d (VIF=31.3)
INFO:src.ml_pipeline:VIF filter: dropped roc_20d (VIF=21.6)
INFO:src.ml_pipeline:VIF filter: dropped bollinger_pctb (VIF=17.5)
INFO:src.ml_pipeline:VIF filter: dropped realized_vol_5d (VIF=11.9)
INFO:src.ml_pipeline:VIF filter removed 6 features: ['garch_cond_vol', 'rsi_14', 'realized_vol_21d', 'roc_20d', 'bollinger_pctb', 'realized_vol_5d']


VIF filter: 17 → 11 features
  Dropped (VIF>10): {'rsi_14', 'roc_20d', 'realized_vol_5d', 'garch_cond_vol', 'bollinger_pctb', 'realized_vol_21d'}


INFO:src.ml_pipeline:Stability selection: 3/17 features selected (threshold=0.60)



Stability Selection: 3/17 features retained (>60% selection)
  Stable features: ['log_return', 'abs_return', 'garch_cond_vol']


INFO:src.ml_pipeline:VIF filter: dropped garch_cond_vol (VIF=123.1)
INFO:src.ml_pipeline:VIF filter: dropped rsi_14 (VIF=69.8)
INFO:src.ml_pipeline:VIF filter: dropped realized_vol_21d (VIF=29.5)
INFO:src.ml_pipeline:VIF filter: dropped roc_20d (VIF=20.9)
INFO:src.ml_pipeline:VIF filter: dropped bollinger_pctb (VIF=16.5)
INFO:src.ml_pipeline:VIF filter: dropped realized_vol_5d (VIF=11.6)
INFO:src.ml_pipeline:VIF filter removed 6 features: ['garch_cond_vol', 'rsi_14', 'realized_vol_21d', 'roc_20d', 'bollinger_pctb', 'realized_vol_5d']
INFO:src.ml_pipeline:Stability selection: 3/17 features selected (threshold=0.60)



--- Feature Selection Comparison (XGBoost, NVDA) ---
       feature_set  n_features  rmse_oos
              full          17  0.241270
      vif_filtered          11  0.245723
stability_selected           3  0.247434

--- Embargo/Purge Note ---
For h=5d forecast: purge last 5 training obs + embargo 5 days after train end
For h=21d forecast: purge last 21 training obs + embargo 21 days
Applied inside walk_forward_predict() via embargo_days parameter


## 3. Walk-Forward Evaluation (All Models × All Tickers)

Run the full model suite across all 20 tickers with 5-day forward vol target.
Each model uses expanding-window walk-forward with quarterly retraining.

In [ ]:
# Full walk-forward evaluation: 6 models × 20 tickers
MODEL_NAMES = ['ridge', 'lasso', 'random_forest', 'xgboost', 'lightgbm', 'knn']

all_results = []      # Per-ticker, per-model metrics
all_predictions = {}  # {(ticker, model): DataFrame of predictions}

for ticker in adj_tickers:
    X, y = build_vol_features(ticker, horizon=5)
    if len(X) < 200:
        print(f'{ticker}: insufficient data ({len(X)} obs), skipping')
        continue

    for model_name in MODEL_NAMES:
        try:
            preds = walk_forward_predict(
                X, y,
                pipeline_factory=lambda n=model_name: make_regression_pipeline(n),
                retrain_freq=RETRAIN_FREQ_DAYS,
                initial_train_ratio=TRAIN_RATIO,
                horizon=5  # 5-day forward vol target → purge/embargo 5 days
            )
            if len(preds) > 0:
                metrics = regression_metrics(preds['y_true'].values, preds['y_pred'].values)
                metrics['ticker'] = ticker
                metrics['model'] = model_name
                metrics['n_predictions'] = len(preds)
                all_results.append(metrics)
                all_predictions[(ticker, model_name)] = preds
        except Exception as e:
            print(f'{ticker}/{model_name}: {e}')
            continue

    print(f'{ticker}: done ({len([r for r in all_results if r["ticker"]==ticker])} models fitted)')

results_df = pd.DataFrame(all_results)
print(f'\nTotal: {len(results_df)} ticker-model combinations evaluated')
print(f'Tickers: {results_df["ticker"].nunique()}, Models: {results_df["model"].nunique()}')

## 4. Model Comparison

Average metrics across tickers per model. Which model family wins for vol forecasting?

In [ ]:
# Average metrics across tickers per model
model_avg = results_df.groupby('model')[['rmse', 'mae', 'mape', 'directional_accuracy']].mean()
model_avg = model_avg.sort_values('rmse')
print('--- Average Metrics Across All Tickers ---')
print(model_avg.round(4))

# Best model per ticker
best_per_ticker = results_df.loc[results_df.groupby('ticker')['rmse'].idxmin()]
print(f'\n--- Best Model Per Ticker (by RMSE) ---')
print(best_per_ticker[['ticker', 'model', 'rmse', 'directional_accuracy']].to_string(index=False))

# Model win count
print(f'\nModel wins: {best_per_ticker["model"].value_counts().to_dict()}')

# Visualization: model comparison bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
model_avg['rmse'].plot.barh(ax=axes[0], color='steelblue')
axes[0].set_title('Average RMSE (lower is better)')
axes[0].set_xlabel('RMSE')
model_avg['directional_accuracy'].plot.barh(ax=axes[1], color='darkorange')
axes[1].set_title('Directional Accuracy % (higher is better)')
axes[1].set_xlabel('DA %')
fig.suptitle('Vol Forecasting: Model Comparison (5-Day Horizon)', fontsize=13)
fig.tight_layout()
save_fig(fig, 'nb07_model_comparison')
plt.show()

## 5. Diebold-Mariano Test & Mincer-Zarnowitz

Statistical comparison of forecast accuracy. DM test: are differences significant?
MZ regression: are forecasts efficient (unbiased and well-calibrated)?

In [ ]:
# Diebold-Mariano test: best model vs Ridge baseline for each ticker
best_model_name = model_avg.index[0]  # lowest RMSE model
dm_results = []
mz_results = []

for ticker in adj_tickers:
    key_best = (ticker, best_model_name)
    key_ridge = (ticker, 'ridge')
    if key_best not in all_predictions or key_ridge not in all_predictions:
        continue

    preds_best = all_predictions[key_best]
    preds_ridge = all_predictions[key_ridge]

    # Align predictions by date
    merged = preds_best.merge(preds_ridge, on='date', suffixes=('_best', '_ridge'))
    if len(merged) < 20:
        continue

    e_best = merged['y_true_best'].values - merged['y_pred_best'].values
    e_ridge = merged['y_true_ridge'].values - merged['y_pred_ridge'].values

    # DM test (h=5 for 5-day forecast horizon → HAC with bandwidth 4)
    dm = diebold_mariano_test(e_best, e_ridge, h=5, loss_fn='squared')
    dm['ticker'] = ticker
    dm_results.append(dm)

    # Mincer-Zarnowitz for best model
    mz = mincer_zarnowitz_test(merged['y_true_best'].values, merged['y_pred_best'].values, h=5)
    mz['ticker'] = ticker
    mz['model'] = best_model_name
    mz_results.append(mz)

dm_df = pd.DataFrame(dm_results)
mz_df = pd.DataFrame(mz_results)

print(f'--- Diebold-Mariano: {best_model_name} vs Ridge ---')
print(f'DM significant (p<0.05): {(dm_df["p_value"] < 0.05).sum()}/{len(dm_df)} tickers')
print(f'Average DM stat: {dm_df["dm_stat"].mean():.3f} (negative = {best_model_name} better)')

print(f'\n--- Mincer-Zarnowitz Efficiency ({best_model_name}) ---')
print(f'Average alpha: {mz_df["alpha"].mean():.4f} (ideal: 0)')
print(f'Average beta: {mz_df["beta"].mean():.4f} (ideal: 1)')
print(f'MZ rejected (p<0.05): {(mz_df["f_pvalue"] < 0.05).sum()}/{len(mz_df)} tickers')

## 5b. QLIKE Loss & BH-FDR on Diebold-Mariano Tests

**QLIKE** (Patton, 2011) — the preferred loss function for volatility forecast
comparison. Unlike RMSE, QLIKE is robust to the choice of volatility proxy
and ranks forecasts consistently:

$$\text{QLIKE} = \frac{1}{T}\sum_t \left[\ln(\hat{\sigma}^2_t) + \frac{\sigma^2_t}{\hat{\sigma}^2_t}\right]$$

**BH-FDR** correction on all pairwise Diebold-Mariano p-values to control
the false discovery rate when testing across 20 tickers simultaneously.

In [ ]:
from src.statistical_tests import benjamini_hochberg

def qlike_loss(y_true, y_pred):
    """QLIKE loss (Patton, 2011) — preferred for volatility forecast evaluation.
    Inputs are annualized volatilities (σ); convert to variance (σ²) for QLIKE.
    QLIKE = mean( ln(σ̂²_t) + σ²_t / σ̂²_t )
    """
    y_pred = np.maximum(y_pred, 1e-10)  # avoid log(0)
    y_true = np.maximum(y_true, 1e-10)
    h_pred = y_pred ** 2
    h_true = y_true ** 2
    return np.mean(np.log(h_pred) + h_true / h_pred)

# ── QLIKE loss per model ──
qlike_results = {}
for model_name in MODEL_NAMES:
    qlikes = []
    for ticker in adj_tickers:
        key = (ticker, model_name)
        if key in all_predictions:
            p = all_predictions[key]
            ql = qlike_loss(p['y_true'].values, p['y_pred'].values)
            qlikes.append(ql)
    if qlikes:
        qlike_results[model_name] = np.mean(qlikes)

print('--- Average QLIKE Loss Per Model (lower is better) ---')
for m, q in sorted(qlike_results.items(), key=lambda x: x[1]):
    print(f'  {m:15s}: {q:.6f}')

# ── BH-FDR correction on DM p-values ──
if not dm_df.empty:
    dm_raw_pvals = dm_df['p_value'].values
    rejected_dm, adj_dm = benjamini_hochberg(dm_raw_pvals, q=0.05)
    dm_df['p_bh_adjusted'] = adj_dm
    dm_df['reject_bh'] = rejected_dm
    
    print(f'\n--- BH-FDR on DM Tests ({best_model_name} vs Ridge) ---')
    print(f'Significant (raw p<0.05): {(dm_raw_pvals < 0.05).sum()}/{len(dm_raw_pvals)}')
    print(f'Significant (BH-adjusted): {rejected_dm.sum()}/{len(rejected_dm)}')
    dm_df.to_csv(TABLES_DIR / 'nb07_diebold_mariano_bh.csv', index=False)

## 6. SHAP Explainability

TreeExplainer for XGBoost/Random Forest: which features drive vol predictions?
Global beeswarm plot + regime-conditional feature importance comparison.

In [ ]:
# SHAP analysis for XGBoost on NVDA (most liquid, longest history)
try:
    import shap

    ticker = 'NVDA'
    X_shap, y_shap = build_vol_features(ticker, horizon=5)
    split = int(len(X_shap) * TRAIN_RATIO)
    X_train_shap, X_test_shap = X_shap.iloc[:split], X_shap.iloc[split:]
    y_train_shap = y_shap.iloc[:split]

    # Fit XGBoost on training data
    pipe = make_regression_pipeline('xgboost')
    pipe.fit(X_train_shap, y_train_shap)

    # Extract the XGBoost model from pipeline for SHAP
    xgb_model = pipe.named_steps['model']
    X_test_scaled = pipe.named_steps['scaler'].transform(X_test_shap)
    X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_shap.columns)

    explainer = shap.TreeExplainer(xgb_model)
    shap_values = explainer.shap_values(X_test_scaled_df)

    # Global feature importance (beeswarm)
    fig, ax = plt.subplots(figsize=(10, 8))
    shap.summary_plot(shap_values, X_test_scaled_df, show=False, max_display=15)
    plt.title(f'SHAP Feature Importance — {ticker} Vol Forecast')
    fig = plt.gcf()
    save_fig(fig, 'nb07_shap_beeswarm')
    plt.show()

    # Mean absolute SHAP importance
    shap_importance = pd.Series(
        np.abs(shap_values).mean(axis=0), index=X_shap.columns
    ).sort_values(ascending=False)
    print(f'\n--- Top 10 Features by Mean |SHAP| ({ticker}) ---')
    for feat, imp in shap_importance.head(10).items():
        print(f'  {feat:25s}: {imp:.6f}')

    # Save SHAP values
    import pickle
    with open(SHAP_VALUES_FILE, 'wb') as f:
        pickle.dump({'shap_values': shap_values, 'feature_names': list(X_shap.columns),
                      'ticker': ticker}, f)
    print(f'\nSaved SHAP values to {SHAP_VALUES_FILE}')

except ImportError:
    print('shap not installed — skipping SHAP analysis')
except Exception as e:
    print(f'SHAP analysis failed: {e}')

## 7. Stacking Ensemble

Meta-learner (Ridge) trained on base model out-of-sample predictions.
Tests if combining diverse models improves over the best individual model.

In [ ]:
# Stacking ensemble for top tickers
stacking_results = []
for ticker in adj_tickers[:5]:  # Top 5 tickers for stacking demonstration
    base_preds = {}
    for model_name in MODEL_NAMES:
        key = (ticker, model_name)
        if key in all_predictions:
            p = all_predictions[key].set_index('date')
            base_preds[model_name] = p['y_pred']

    if len(base_preds) < 3:
        continue

    base_df = pd.DataFrame(base_preds)
    y_true_series = all_predictions[(ticker, 'ridge')].set_index('date')['y_true']

    # Align all predictions
    aligned = base_df.join(y_true_series, how='inner').dropna()
    if len(aligned) < 50:
        continue

    X_stack = aligned[list(base_preds.keys())]
    y_stack = aligned['y_true']

    # Train stacking meta-learner on first 70%, evaluate on rest
    split = int(len(X_stack) * 0.7)
    stack_pipe = build_stacking_ensemble(X_stack.iloc[:split], y_stack.iloc[:split])
    stack_pred = stack_pipe.predict(X_stack.iloc[split:])
    stack_metrics = regression_metrics(y_stack.iloc[split:].values, stack_pred)
    stack_metrics['ticker'] = ticker
    stack_metrics['model'] = 'stacking'
    stacking_results.append(stack_metrics)

    # Compare with best individual
    best_individual = results_df[results_df['ticker'] == ticker].sort_values('rmse').iloc[0]
    print(f'{ticker}: Stacking RMSE={stack_metrics["rmse"]:.6f} vs '
          f'Best({best_individual["model"]}) RMSE={best_individual["rmse"]:.6f}')

if stacking_results:
    stack_df = pd.DataFrame(stacking_results)
    results_df = pd.concat([results_df, stack_df], ignore_index=True)

## 8. Save Outputs

In [ ]:
# Save all predictions (for NB10 hybrid model + NB11 portfolio)
all_pred_list = []
for (ticker, model_name), preds in all_predictions.items():
    p = preds.copy()
    p['ticker'] = ticker
    p['model'] = model_name
    all_pred_list.append(p)
if all_pred_list:
    all_pred_df = pd.concat(all_pred_list, ignore_index=True)
    all_pred_df.to_parquet(VOL_FORECAST_FILE)
    print(f'Saved predictions: {VOL_FORECAST_FILE} ({all_pred_df.shape})')

# Save model comparison table
results_df.to_csv(MODEL_COMPARISON_FILE, index=False)
print(f'Saved comparison: {MODEL_COMPARISON_FILE} ({results_df.shape})')

# Save DM and MZ test results
if not dm_df.empty:
    dm_df.to_csv(TABLES_DIR / 'nb07_diebold_mariano.csv', index=False)
if not mz_df.empty:
    mz_df.to_csv(TABLES_DIR / 'nb07_mincer_zarnowitz.csv', index=False)

print('\n--- NB07 Complete ---')
print(f'  Tickers: {results_df["ticker"].nunique()}')
print(f'  Models: {results_df["model"].nunique()} ({", ".join(results_df["model"].unique())})')
print(f'  Best overall model: {model_avg.index[0]} (avg RMSE={model_avg["rmse"].iloc[0]:.6f})')
print(f'\nReady for: NB09 (DL comparison), NB10 (hybrid model + audit)')